# Model Comparison — All 4 Experiments (GLOBAL — All 9 Devices)

Loads `metrics.json` from each global model and produces a **side-by-side comparison**.

| Model | Features Used | Classifier |
|---|---|---|
| Model 1 | All 115 features (baseline) | Random Forest |
| Model 2 | Variance Threshold subset | Random Forest |
| Model 3 | Pearson Correlation subset | Random Forest |
| Model 4 | Mutual Information top-40 | Random Forest |

**Metrics (from PPT Slide 8):** Accuracy, Precision, Recall, F1, FPR, Inference Time, Feature Count

In [ ]:
from pathlib import Path
import json
import pandas as pd

PROJECT_ROOT = Path(r'c:\Users\kulsu\Downloads\detection+of+iot+botnet+attacks+n+baiot')
MODELS_DIR = PROJECT_ROOT / 'preprocessed' / 'global_models'

models = [
    ('Model1_AllFeatures',        'All 115 Features (Baseline)'),
    ('Model2_VarianceThreshold',  'Variance Threshold'),
    ('Model3_PearsonCorrelation', 'Pearson Correlation'),
    ('Model4_MutualInformation',  'Mutual Information'),
]

rows = []
for folder, display_name in models:
    mp = MODELS_DIR / folder / 'metrics.json'
    if not mp.exists():
        print(f'⚠️  Not found: {mp.name} — run that model notebook first')
        continue
    with open(mp) as f:
        m = json.load(f)
    rows.append({
        'Model': display_name,
        'Features': m.get('feature_count', '?'),
        'Accuracy': m.get('accuracy'),
        'Precision (macro)': m.get('macro_precision'),
        'Recall (macro)': m.get('macro_recall'),
        'F1 (macro)': m.get('macro_f1'),
        'FPR (benign)': m.get('false_positive_rate'),
        'Inference Time (s)': m.get('total_inference_time'),
        'Training Time (s)': m.get('training_time_s'),
        'Test Rows': m.get('test_rows'),
    })

df_cmp = pd.DataFrame(rows)
print(f"\n{'='*85}")
print(f'  FEATURE SELECTION COMPARISON — GLOBAL MODEL (All 9 IoT Devices)')
print(f"{'='*85}")
df_cmp

In [ ]:
# Formatted as percentages
df_fmt = df_cmp.copy()
for col in ['Accuracy', 'Precision (macro)', 'Recall (macro)', 'F1 (macro)', 'FPR (benign)']:
    if col in df_fmt.columns:
        df_fmt[col] = df_fmt[col].apply(lambda x: f'{x*100:.2f}%' if x is not None else 'N/A')
for col in ['Inference Time (s)', 'Training Time (s)']:
    if col in df_fmt.columns:
        df_fmt[col] = df_fmt[col].apply(lambda x: f'{x:.2f}' if x is not None else 'N/A')

print('\nFormatted Comparison Table:')
print(df_fmt.to_string(index=False))

In [ ]:
# Per-device accuracy comparison across models
print('\n--- Per-Device Accuracy Across Models ---')
dev_rows = []
for folder, display_name in models:
    da_path = MODELS_DIR / folder / 'per_device_accuracy.csv'
    if not da_path.exists():
        continue
    da = pd.read_csv(da_path)
    for _, r in da.iterrows():
        dev_rows.append({'Model': display_name, 'Device': r['Device'], 'Accuracy': r['Accuracy']})

if dev_rows:
    df_dev = pd.DataFrame(dev_rows)
    df_pivot = df_dev.pivot(index='Device', columns='Model', values='Accuracy')
    # Reorder columns to match model order
    col_order = [n for _, n in models if n in df_pivot.columns]
    df_pivot = df_pivot[col_order]
    print(df_pivot.round(4).to_string())
else:
    print('No per-device data available yet.')

In [ ]:
# Save comparison
MODELS_DIR.mkdir(parents=True, exist_ok=True)
df_cmp.to_csv(MODELS_DIR / 'global_model_comparison.csv', index=False)
print(f'✅ Saved: {MODELS_DIR / "global_model_comparison.csv"}')

In [ ]:
# Visualization
try:
    import matplotlib.pyplot as plt
    
    fig, axes = plt.subplots(1, 3, figsize=(17, 5))
    model_names = df_cmp['Model'].tolist()
    colors = ['#3498db', '#2ecc71', '#e67e22', '#9b59b6']
    x = range(len(model_names))
    
    # 1. Accuracy & F1
    ax = axes[0]
    w = 0.35
    bars1 = ax.bar([i-w/2 for i in x], df_cmp['Accuracy']*100, w, label='Accuracy %', color=colors)
    bars2 = ax.bar([i+w/2 for i in x], df_cmp['F1 (macro)']*100, w, label='Macro F1 %', color=colors, alpha=0.6)
    ax.set_xticks(x)
    ax.set_xticklabels([n.replace(' (Baseline)', '') for n in model_names], rotation=20, ha='right', fontsize=7)
    ax.set_ylabel('Score (%)')
    ax.set_title('Accuracy & Macro F1')
    ax.legend(fontsize=8)
    ax.set_ylim(max(0, df_cmp['Accuracy'].min()*100 - 5), 101)
    
    # 2. Feature count
    ax = axes[1]
    ax.bar(x, df_cmp['Features'], color=colors)
    ax.set_xticks(x)
    ax.set_xticklabels([n.replace(' (Baseline)', '') for n in model_names], rotation=20, ha='right', fontsize=7)
    ax.set_ylabel('Feature Count')
    ax.set_title('Features Used')
    for i, v in enumerate(df_cmp['Features']):
        ax.text(i, v+1, str(v), ha='center', fontsize=10, fontweight='bold')
    
    # 3. Inference time
    ax = axes[2]
    ax.bar(x, df_cmp['Inference Time (s)'], color=colors)
    ax.set_xticks(x)
    ax.set_xticklabels([n.replace(' (Baseline)', '') for n in model_names], rotation=20, ha='right', fontsize=7)
    ax.set_ylabel('Seconds')
    ax.set_title('Inference Time (all test data)')
    
    plt.suptitle('Feature Selection Comparison — Global Model (9 IoT Devices)', fontsize=13, fontweight='bold')
    plt.tight_layout()
    fig.savefig(MODELS_DIR / 'global_comparison_chart.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: global_comparison_chart.png')
except ImportError:
    print('matplotlib not available — skipping charts')

---
## Key Questions Answered (PPT Slide 7)

1. **Which technique keeps the most useful information?** → Compare F1 scores
2. **How far can the feature count be reduced?** → Compare Features column
3. **Does accuracy rise, fall, or stay similar?** → Compare Accuracy column
4. **Which gives the best performance–efficiency balance?** → Compare F1 vs Inference Time

### Research Questions (PPT Slide 2)
- **RQ1:** How does feature selection affect IoT botnet detection performance?
- **RQ2:** Which technique gives the best balance of detection performance and computational efficiency?
- **RQ3:** Can the selected models demonstrate detection on traffic from a physical IoT testbed?